# 🤖 Notebook 05 — Fraud Risk Model (Binary Classification)

**Objective**: Train an ML model (XGBoost/LightGBM) to predict `is_aml`.

In [1]:
import pandas as pd
import numpy as np
import os
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve, auc
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

FEATURE_DIR = os.path.join('..', 'data', 'features')
MODEL_DIR = os.path.join('..', 'outputs', 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

txn = pd.read_parquet(os.path.join(FEATURE_DIR, 'transactions_selected.parquet'))
drop_cols = ['transaction_id', 'customer_cif_id', 'customer_account_number', 'device_id_fingerprint', 'wallet_account_id', 'is_aml', 'aml_typology', 'fraud_intensity_score', 'fis_band']
features = [c for c in txn.columns if c not in drop_cols]

X = txn[features]
y = txn['is_aml']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Training set shape:", X_train.shape)

Training set shape: (309256, 50)


In [2]:
# Train LightGBM Model
clf = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05, max_depth=8, random_state=42, n_jobs=-1, class_weight='balanced')
clf.fit(X_train, y_train, eval_set=[(X_test, y_test)])

# Evaluate
y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

print("ROC AUC:", roc_auc_score(y_test, y_proba))
print("Classification Report:\n", classification_report(y_test, y_pred))

# Save Model
joblib.dump(clf, os.path.join(MODEL_DIR, 'lgbm_is_aml.pkl'))
print("Saved Fraud Risk Model.")

[LightGBM] [Info] Number of positive: 58283, number of negative: 250973
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.011884 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 8764
[LightGBM] [Info] Number of data points in the train set: 309256, number of used features: 50
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


ROC AUC: 0.9290881190914146
Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.85      0.90     62743
           1       0.58      0.86      0.69     14571

    accuracy                           0.85     77314
   macro avg       0.77      0.86      0.80     77314
weighted avg       0.89      0.85      0.86     77314

Saved Fraud Risk Model.
